In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv("../Dataset/cleaned_data.csv")
print("missing values are=",df["AQI"].isna().sum())

missing values are= 9139


In [3]:
df['Datetime'] = pd.to_datetime(df['Datetime'])

# --- Step 2: Cold Start Fix ---
# Remove leading empty rows before sensor started recording
first_reading_row = df['AQI'].first_valid_index()
start_time = df.loc[first_reading_row, 'Datetime']

print(f"Original total rows: {len(df)}")
print(f"Sensor first recording: {start_time}")

df = df.loc[first_reading_row:].reset_index(drop=True)

# --- NEW STEP 2.5: Fix the 720 Missing Timestamps (Nov 2017 Table Gap) ---
# Create a complete, unbroken hourly calendar from start_time to 2023-12-31 23:00:00
df = df.sort_values('Datetime').drop_duplicates(subset=['Datetime'])
full_hourly_index = pd.date_range(start=df['Datetime'].min(), end=df['Datetime'].max(), freq='h')
df = df.set_index('Datetime').reindex(full_hourly_index).reset_index()
df = df.rename(columns={'index': 'Datetime'})

# Re-populate helper columns for any newly inserted timestamp rows
df['Month_Year'] = df['Datetime'].dt.strftime('%B-%Y')
df['Day'] = df['Datetime'].dt.day
df['Time'] = df['Datetime'].dt.strftime('%H:%M:%S')

print(f"Rows after timeline repair (0 missing hours): {len(df)}")
print(f"Missing values to handle (including 720 Nov-2017 hours): {df['AQI'].isna().sum()}")

# --- Step 3: Layered Missing Value Strategy ---
# Layer 1: Forward fill short gaps up to 6 hours
df['AQI_Clean'] = df['AQI'].ffill(limit=6)

# Layer 2: Linear interpolation for medium gaps up to 24 hours
df['AQI_Clean'] = df['AQI_Clean'].interpolate(
    method='linear',
    limit=24,
    limit_direction='both'
)

# Layer 3: Monthly average for longer gaps
df['YearMonth'] = df['Datetime'].dt.to_period('M')
monthly_avg = df.groupby('YearMonth')['AQI_Clean'].transform('mean')
df['AQI_Clean'] = df['AQI_Clean'].fillna(monthly_avg)

# Layer 3.5: Calendar-Month average (specifically catches full missing months like Nov 2017
# by using the average of all other Novembers instead of the lower global mean!)
cal_month_avg = df.groupby(df['Datetime'].dt.month)['AQI_Clean'].transform('mean')
df['AQI_Clean'] = df['AQI_Clean'].fillna(cal_month_avg)

# Layer 4: Global mean as final fallback
df['AQI_Clean'] = df['AQI_Clean'].fillna(df['AQI_Clean'].mean())

# --- Step 4: Finalize & Layer 5 Realistic Floor (40) ---
df = df.drop(columns=['AQI', 'YearMonth'])
df = df.rename(columns={'AQI_Clean': 'AQI'})

REALISTIC_FLOOR = 40
fault_mask = df['AQI'] < REALISTIC_FLOOR
print(f"Flagging {fault_mask.sum()} rows as sensor faults / interpolation artifacts")

df.loc[fault_mask, 'AQI'] = np.nan
df['AQI'] = df['AQI'].ffill(limit=6)
df['AQI'] = df['AQI'].interpolate(method='linear', limit=24, limit_direction='both')
df['YearMonth'] = df['Datetime'].dt.to_period('M')
df['AQI'] = df['AQI'].fillna(df.groupby('YearMonth')['AQI'].transform('mean'))
df['AQI'] = df['AQI'].fillna(df.groupby(df['Datetime'].dt.month)['AQI'].transform('mean'))
df['AQI'] = df['AQI'].fillna(df['AQI'].mean())
df = df.drop(columns=['YearMonth'])

print("\n=== CLEANING SUMMARY ===")
print(f"Total rows: {len(df)} (Should be 54406)")
print(f"Date range: {df['Datetime'].min()} to {df['Datetime'].max()}")
print(f"Missing values remaining: {df['AQI'].isna().sum()}")

df.to_csv("../Dataset/final_clean_aqi.csv", index=False)
print("Saved fixed dataset to ../Dataset/final_clean_aqi.csv")

Original total rows: 55584
Sensor first recording: 2017-10-17 02:00:00
Rows after timeline repair (0 missing hours): 54406
Missing values to handle (including 720 Nov-2017 hours): 7961
Flagging 351 rows as sensor faults / interpolation artifacts

=== CLEANING SUMMARY ===
Total rows: 54406 (Should be 54406)
Date range: 2017-10-17 02:00:00 to 2023-12-31 23:00:00
Missing values remaining: 0
Saved fixed dataset to ../Dataset/final_clean_aqi.csv
